#Chat Metrics

##Data Collection

In [1]:
import pandas            as pd
import numpy             as np
import matplotlib.pyplot as plt
import seaborn           as sns
import graphviz

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
  accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  roc_auc_score,
  classification_report
)
from sklearn import tree

#Read in from hugging face
df = pd.read_parquet('https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/chat_metrics.parquet')

##Processing

In [2]:
top_5 = [
  'month_sin',
  'primary_category_navigation',
  'output_tokens',
  'answer_sentiment',
  'hour_sin'
]

top_7 = [
  'month_sin',
  'primary_category_navigation',
  'output_tokens',
  'answer_sentiment',
  'hour_sin',
  'model_calls',
  'primary_category_sunport_amenities'
]

top_10 = [
  'month_sin',
  'primary_category_navigation',
  'output_tokens',
  'answer_sentiment',
  'hour_sin',
  'model_calls',
  'primary_category_sunport_amenities',
  'day_sin',
  'question_length',
  'total_tokens'
]

all_features = [

  #Time
  'day_sin',
  'day_cos',
  'hour_sin',
  'hour_cos',
  'month_sin',
  'month_cos',

  #Performance
  'processing_time_seconds',
  'total_tokens',
  'input_tokens',
  'output_tokens',
  'model_calls',
  'tool_calls_count',

  #Q&A
  'question_length',
  'answer_length',

  #Maps
  'has_geolocation',

  #Primary Category
  'primary_category_greetings',
  'primary_category_sunport_amenities',
  'primary_category_navigation',
  'primary_category_airline_logistics',
  'primary_category_general_info',

  #Selected Agent
  'selected_agentreporter',
  'selected_agentplanner',
  'selected_agentlocation',
  'selected_agentlocation_fallback_to_reporter',
  'selected_agentbroad_search_synthesis',
  'selected_agentbroad_search_passthrough',

  #Sentiment
  'question_sentiment',
  'answer_sentiment'
]

X = df[all_features]
y = df['satisfaction']

X_train, X_test, y_train, y_test = train_test_split(
  X,
  y,
  test_size=0.20,
  random_state=42,
  stratify=y
)

features = {
  'All Features': all_features,
  'Top 5': top_5,
  'Top 7': top_7,
  'Top 10': top_10
}

results = []

for name, feature in features.items():

  #Model
  xgb = XGBClassifier(
    n_estimators=300,   #Boosting rounds/trees
    learning_rate=0.20, #Controls how much each new tree contributest to model
    max_depth=1,        #Max deptho or each tree
    random_state=42,
    objective = 'binary:logistic', # Tells XGBoost to do binary classification
    eval_metric='logloss'
  )

  #Fit
  xgb.fit(X_train[feature], y_train)

  #Predict
  y_pred_xgb = xgb.predict(X_test[feature])

  #Probability
  y_prob_xgb = xgb.predict_proba(X_test[feature])[:,1]

  results.append({
    'Feature Set': name,
    'Features': len(feature),
    'Accuracy': accuracy_score(y_test, y_pred_xgb),
    'Precision': precision_score(y_test, y_pred_xgb),
    'Recall': recall_score(y_test, y_pred_xgb),
    'F1': f1_score(y_test, y_pred_xgb),
    'ROC-AUC': roc_auc_score(y_test, y_prob_xgb)
  })

results_df = pd.DataFrame(results)
results_df

,Feature Set,Features,Accuracy,Precision,Recall,F1,ROC-AUC
0,All Features,28,0.747126,0.785714,0.717391,0.750000,0.828738
1,Top 5,5,0.758621,0.777778,0.760870,0.769231,0.829268
2,Top 7,7,0.758621,0.777778,0.760870,0.769231,0.838547
3,Top 10,10,0.781609,0.800000,0.782609,0.791209,0.856310
